# Deterministic environment trace

**Reproduction type:** software trace · **Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds

This trace uses the OHLC and spread values from the first 180 bars of the bundled US100 example. To make a single isolated session, it assigns those bars a continuous fixture clock beginning at 08:00; the notebook does not claim to replay the source timestamps. A fixed long action is applied once, and the resulting fill, stop, equity, and reward are asserted. No trained checkpoint is used.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from simulation.environment.env import TradingEnv

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
source = ROOT / "data/examples/US100.cash_M1_202412300105_202501020334.csv"
raw = pd.read_csv(source, sep="\t").head(180)
fixture_index = pd.date_range("2025-01-02 08:00", periods=len(raw), freq="min")
close = raw["<CLOSE>"].to_numpy(dtype=float)
bars = pd.DataFrame(
    {
        "open": raw["<OPEN>"].to_numpy(dtype=float),
        "high": raw["<HIGH>"].to_numpy(dtype=float),
        "low": raw["<LOW>"].to_numpy(dtype=float),
        "close": close,
        "volume": raw["<TICKVOL>"].to_numpy(dtype=float),
        "spread": raw["<SPREAD>"].to_numpy(dtype=float) * 0.01,
        "session_id": 0,
    },
    index=fixture_index,
)
features = pd.DataFrame(
    {
        "close": close,
        "last_swing_high": close + 10.0,
        "last_swing_low": close - 10.0,
        "london_high": close + 20.0,
        "london_low": close - 20.0,
        "asian_high": close + 20.0,
        "asian_low": close - 20.0,
        "volume_spike": 1.0,
    },
    index=fixture_index,
)
env = TradingEnv(
    bars,
    features,
    obs_window=10,
    initial_balance=10_000.0,
    risk_frac_range=(0.01, 0.01),
    rr_ratio_range=(2.0, 2.0),
    min_sl_atr_mult=0.0,
    min_sl_points=0.0,
    block_overnight=False,
    max_episode_steps=150,
)
observation, _ = env.reset(seed=42)
print({"sample bars": len(bars), "observation keys": sorted(observation)})

In [ ]:
observation, reward, terminated, truncated, info = env.step(1)
assert env.position is not None
trace = {
    "decision_bar": env.step_idx - 1,
    "fill_price": float(env.position.entry_price),
    "stop_price": float(env.position.sl_price),
    "target_price": float(env.position.tp_price),
    "equity": float(env.account.equity),
    "info_equity": float(info["equity"]),
    "reward": float(reward),
    "entry_count": int(env._entry_diag["opened"]),
}
print(trace)
assert trace["decision_bar"] == 10
assert trace["entry_count"] == 1
assert np.isclose(trace["fill_price"], 21489.879, atol=1e-6)
assert np.isclose(trace["stop_price"], 21479.45, atol=1e-6)
assert np.isclose(trace["target_price"], 21510.737, atol=1e-6)
assert np.isclose(trace["equity"], 10_000.0, atol=1e-8)
assert trace["equity"] == trace["info_equity"]
assert np.isclose(trace["reward"], 0.0, atol=1e-8)
assert not terminated and not truncated
print(
    "deterministic environment trace passed; this is software behavior evidence, not a research result."
)

## How to interpret the trace

The source OHLC and spread are preserved, but timestamps and feature context are deliberately constructed for this short deterministic session. This validates transition wiring and a known fill; it is not a historical replay or strategy result.

See [environment step](../architecture/runtime/environment_step), [TradingEnv API](../api/domains/trading), [tutorial explanation](../learn/tutorials/04_environment.ipynb), and [limitations](../research/limitations).